# AuraGateway current CUDA 12.9 harness recovery materializer

This replacement materializer handles Kaggle's auto-expanded source ZIP representation.

Use exactly:

- **Kaggle title:** `ag-harness-materializer-cu129-v1`
- **Input:** `ag-harness-426f57d-v1-input` Version 1
- **Accelerator:** None
- **Internet:** Off
- **Secrets:** None
- **Execution:** Save Version → Save & Run All

The notebook reconstructs the exact deterministic ZIP from the expanded tree, verifies its SHA-256
against the frozen source receipt, materializes the current harness, and emits the same producer
contract required by the metadata-only inspection notebook.


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "ag-harness-materializer-cu129-v1"
DATASET_NAME = "ag-harness-426f57d-v1-input"
DATASET_OWNER = "kabomolefe"
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()

EXPECTED_SOURCE_COMMIT = "426f57dd11dddc2fb8e5a703721c2189abc7a0ff"
EXPECTED_ARCHIVE_NAME = "ag-harness-426f57d-v1.zip"
EXPECTED_EXPANDED_DIRECTORY = "ag-harness-426f57d-v1"
EXPECTED_OUTPUT_DIRECTORY = "auragateway_qualification_harness_426f57d_v1"
EXPECTED_MATERIALIZATION_RECEIPT_NAME = (
    "ag_harness_materialization_receipt_cu129_v1.json"
)
EXPECTED_CONTROL_FILES = (
    "source_inventory.json",
    "source_packaging_receipt.json",
    "sha256_manifest.json",
)

PRODUCER_OUTPUT_DIRECTORY = "ag_harness_materializer_cu129_v1_output"
FINAL_BUNDLE_ROOT = WORK_ROOT / PRODUCER_OUTPUT_DIRECTORY
STAGING_BUNDLE_ROOT = WORK_ROOT / ".ag_harness_materializer_cu129_v1_staging"
STAGING_ARCHIVE_PATH = STAGING_BUNDLE_ROOT / EXPECTED_ARCHIVE_NAME
STAGING_HARNESS_ROOT = STAGING_BUNDLE_ROOT / EXPECTED_OUTPUT_DIRECTORY
STAGING_RECEIPT_PATH = (
    STAGING_BUNDLE_ROOT / EXPECTED_MATERIALIZATION_RECEIPT_NAME
)

FAILURE_DIRECTORY = WORK_ROOT / "ag_harness_materializer_cu129_v1_failure"
FAILURE_ZIP_PATH = WORK_ROOT / "ag-harness-materializer-cu129-v1-failure.zip"

MAXIMUM_FILES = 5_000
MAXIMUM_TOTAL_BYTES = 100 * 1024 * 1024
ZIP_TIMESTAMP = (1980, 1, 1, 0, 0, 0)
ARCHIVE_SUFFIXES = (
    ".zip",
    ".tar",
    ".tgz",
    ".gz",
    ".bz2",
    ".xz",
    ".7z",
    ".whl",
)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def bytes_sha256(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def normalized_relative_path(value: str) -> PurePosixPath:
    path = PurePosixPath(value)
    if (
        path.is_absolute()
        or not path.parts
        or ".." in path.parts
        or "\\" in value
        or value.startswith("./")
        or path.as_posix() != value
    ):
        raise RuntimeError("source inventory contains an unsafe relative path")
    return path


def load_canonical_json(path: Path) -> object:
    raw = path.read_bytes()
    try:
        payload = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"invalid JSON control: {path.name}") from exc
    expected = canonical_json(payload).encode("utf-8")
    if raw != expected:
        raise RuntimeError(f"JSON control is not canonical: {path.name}")
    return payload


def resolve_dataset_root() -> Path:
    candidates = (
        INPUT_ROOT / DATASET_NAME,
        INPUT_ROOT / "datasets" / DATASET_OWNER / DATASET_NAME,
    )
    observed = tuple(
        candidate.resolve()
        for candidate in candidates
        if candidate.is_dir() and not candidate.is_symlink()
    )
    if len(observed) != 1:
        raise RuntimeError(
            "expected exactly one attached expanded harness source dataset "
            f"but observed {len(observed)}"
        )
    dataset_root = observed[0]
    if INPUT_ROOT not in dataset_root.parents:
        raise RuntimeError("harness source dataset escaped /kaggle/input")
    return dataset_root


def validate_dataset_topology(dataset_root: Path) -> Path:
    expected_names = {
        EXPECTED_EXPANDED_DIRECTORY,
        *EXPECTED_CONTROL_FILES,
    }
    observed_names: set[str] = set()

    for path in dataset_root.iterdir():
        if path.is_symlink():
            raise RuntimeError("source dataset contains a symbolic link")
        observed_names.add(path.name)

    if observed_names != expected_names:
        raise RuntimeError(
            "expanded source dataset top-level set drifted: "
            f"expected={sorted(expected_names)} observed={sorted(observed_names)}"
        )

    expanded_root = dataset_root / EXPECTED_EXPANDED_DIRECTORY
    if not expanded_root.is_dir() or expanded_root.is_symlink():
        raise RuntimeError("expected expanded source directory is missing or symbolic")

    for name in EXPECTED_CONTROL_FILES:
        path = dataset_root / name
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(f"source control is missing or non-regular: {name}")

    return expanded_root


def load_and_validate_controls(
    dataset_root: Path,
) -> tuple[dict[str, object], list[dict[str, object]], dict[str, str]]:
    receipt_raw = (dataset_root / "source_packaging_receipt.json").read_bytes()
    inventory_raw = (dataset_root / "source_inventory.json").read_bytes()
    sha_manifest_raw = (dataset_root / "sha256_manifest.json").read_bytes()

    receipt_payload = load_canonical_json(
        dataset_root / "source_packaging_receipt.json"
    )
    inventory_payload = load_canonical_json(dataset_root / "source_inventory.json")
    sha_manifest_payload = load_canonical_json(dataset_root / "sha256_manifest.json")

    if not isinstance(receipt_payload, dict):
        raise RuntimeError("source receipt root must be one object")
    if not isinstance(inventory_payload, list):
        raise RuntimeError("source inventory root must be one array")
    if not isinstance(sha_manifest_payload, dict):
        raise RuntimeError("source SHA-256 manifest root must be one object")

    receipt = dict(receipt_payload)
    inventory = list(inventory_payload)

    sha_manifest: dict[str, str] = {}
    for key, value in sha_manifest_payload.items():
        if not isinstance(key, str) or not isinstance(value, str):
            raise RuntimeError("source SHA-256 manifest contains an invalid entry")
        sha_manifest[key] = value

    expected_receipt_values = {
        "status": "CURRENT_CU129_HARNESS_SOURCE_PACKAGED",
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "archive_name": EXPECTED_ARCHIVE_NAME,
        "input_dataset_name": DATASET_NAME,
        "output_directory": EXPECTED_OUTPUT_DIRECTORY,
        "materialization_receipt_name": EXPECTED_MATERIALIZATION_RECEIPT_NAME,
    }
    for key, value in expected_receipt_values.items():
        if receipt.get(key) != value:
            raise RuntimeError(f"source receipt field drifted: {key}")

    safety = receipt.get("safety")
    if not isinstance(safety, dict):
        raise RuntimeError("source receipt safety contract is missing")
    expected_safety = {
        "network_access_performed": False,
        "package_installation_performed": False,
        "gpu_execution_performed": False,
        "model_loaded": False,
        "tokenizer_loaded": False,
        "worker_started": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "authorization_issued": False,
        "credentials_present": False,
        "customer_data_present": False,
        "external_spend": 0,
    }
    if any(safety.get(key) != value for key, value in expected_safety.items()):
        raise RuntimeError("source receipt safety contract drifted")

    expected_manifest_keys = {
        EXPECTED_ARCHIVE_NAME,
        "source_inventory.json",
        "source_packaging_receipt.json",
    }
    if set(sha_manifest) != expected_manifest_keys:
        raise RuntimeError("source SHA-256 manifest key set drifted")

    if sha_manifest["source_inventory.json"] != bytes_sha256(inventory_raw):
        raise RuntimeError("source inventory identity drifted from SHA-256 manifest")
    if sha_manifest["source_packaging_receipt.json"] != bytes_sha256(receipt_raw):
        raise RuntimeError("source receipt identity drifted from SHA-256 manifest")
    if receipt.get("inventory_sha256") != bytes_sha256(inventory_raw):
        raise RuntimeError("source receipt inventory identity drifted")
    if receipt.get("archive_sha256") != sha_manifest[EXPECTED_ARCHIVE_NAME]:
        raise RuntimeError("source archive identity authorities disagree")

    if bytes_sha256(sha_manifest_raw) != file_sha256(
        dataset_root / "sha256_manifest.json"
    ):
        raise RuntimeError("source SHA-256 manifest read instability detected")

    return receipt, inventory, sha_manifest


def validate_inventory(
    inventory: list[dict[str, object]],
    receipt: dict[str, object],
) -> dict[str, dict[str, object]]:
    if not inventory or len(inventory) > MAXIMUM_FILES:
        raise RuntimeError("source inventory file count is outside the approved budget")

    by_path: dict[str, dict[str, object]] = {}
    ordered_paths: list[str] = []
    total_bytes = 0

    for raw_entry in inventory:
        if not isinstance(raw_entry, dict):
            raise RuntimeError("source inventory contains an invalid entry")

        required_keys = {
            "path",
            "git_blob_sha",
            "sha256",
            "size_bytes",
            "executable",
        }
        if set(raw_entry) != required_keys:
            raise RuntimeError("source inventory entry key set drifted")

        path_value = raw_entry["path"]
        git_blob_sha = raw_entry["git_blob_sha"]
        sha256 = raw_entry["sha256"]
        size_bytes = raw_entry["size_bytes"]
        executable = raw_entry["executable"]

        if not isinstance(path_value, str):
            raise RuntimeError("source inventory path is invalid")
        relative_path = normalized_relative_path(path_value).as_posix()
        if relative_path.lower().endswith(ARCHIVE_SUFFIXES):
            raise RuntimeError("source inventory contains a nested archive")
        if relative_path in by_path:
            raise RuntimeError("source inventory contains duplicate paths")

        if (
            not isinstance(git_blob_sha, str)
            or len(git_blob_sha) != 40
            or any(character not in "0123456789abcdef" for character in git_blob_sha)
        ):
            raise RuntimeError("source inventory Git blob id is invalid")
        if (
            not isinstance(sha256, str)
            or len(sha256) != 64
            or any(character not in "0123456789abcdef" for character in sha256)
        ):
            raise RuntimeError("source inventory SHA-256 is invalid")
        if (
            not isinstance(size_bytes, int)
            or isinstance(size_bytes, bool)
            or size_bytes < 0
        ):
            raise RuntimeError("source inventory size is invalid")
        if not isinstance(executable, bool):
            raise RuntimeError("source inventory executable flag is invalid")

        entry = {
            "path": relative_path,
            "git_blob_sha": git_blob_sha,
            "sha256": sha256,
            "size_bytes": size_bytes,
            "executable": executable,
        }
        by_path[relative_path] = entry
        ordered_paths.append(relative_path)
        total_bytes += size_bytes

        if total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("source inventory exceeds the byte budget")

    if ordered_paths != sorted(ordered_paths):
        raise RuntimeError("source inventory path order is not canonical")
    if receipt.get("file_count") != len(inventory):
        raise RuntimeError("source receipt file count drifted")
    if receipt.get("total_bytes") != total_bytes:
        raise RuntimeError("source receipt total bytes drifted")

    identity_entries = [
        {
            "path": entry["path"],
            "sha256": entry["sha256"],
            "size_bytes": entry["size_bytes"],
        }
        for entry in inventory
    ]
    directory_sha256 = bytes_sha256(
        canonical_json(identity_entries).encode("utf-8")
    )
    if receipt.get("directory_sha256") != directory_sha256:
        raise RuntimeError("source directory identity drifted")

    return by_path


def inspect_expanded_tree(
    expanded_root: Path,
    by_path: dict[str, dict[str, object]],
) -> None:
    observed_paths: list[str] = []
    observed_total_bytes = 0

    for path in sorted(expanded_root.rglob("*"), key=lambda item: item.as_posix()):
        if path.is_symlink():
            raise RuntimeError("expanded source tree contains a symbolic link")

        metadata = path.stat()
        if path.is_dir():
            continue
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError("expanded source tree contains a non-regular member")

        relative_path = path.relative_to(expanded_root).as_posix()
        normalized_relative_path(relative_path)
        if relative_path.lower().endswith(ARCHIVE_SUFFIXES):
            raise RuntimeError("expanded source tree contains a nested archive")

        expected = by_path.get(relative_path)
        if expected is None:
            raise RuntimeError(f"expanded source tree contains an extra file: {relative_path}")
        if metadata.st_size != expected["size_bytes"]:
            raise RuntimeError(f"expanded source file size drifted: {relative_path}")
        if file_sha256(path) != expected["sha256"]:
            raise RuntimeError(f"expanded source file identity drifted: {relative_path}")

        observed_paths.append(relative_path)
        observed_total_bytes += metadata.st_size

        if len(observed_paths) > MAXIMUM_FILES:
            raise RuntimeError("expanded source tree exceeds the file-count budget")
        if observed_total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("expanded source tree exceeds the byte budget")

    if observed_paths != sorted(by_path):
        missing = sorted(set(by_path) - set(observed_paths))
        extra = sorted(set(observed_paths) - set(by_path))
        raise RuntimeError(
            f"expanded source path set drifted: missing={missing[:10]} extra={extra[:10]}"
        )


def reconstruct_exact_archive(
    expanded_root: Path,
    inventory: list[dict[str, object]],
    expected_archive_sha256: str,
) -> None:
    with zipfile.ZipFile(
        STAGING_ARCHIVE_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
        strict_timestamps=True,
    ) as archive:
        for entry in inventory:
            relative_path = str(entry["path"])
            source_path = expanded_root.joinpath(
                *normalized_relative_path(relative_path).parts
            )
            payload = source_path.read_bytes()

            info = zipfile.ZipInfo(relative_path, date_time=ZIP_TIMESTAMP)
            info.create_system = 3
            info.compress_type = zipfile.ZIP_DEFLATED
            mode = 0o100755 if entry["executable"] else 0o100644
            info.external_attr = mode << 16

            archive.writestr(
                info,
                payload,
                compress_type=zipfile.ZIP_DEFLATED,
                compresslevel=9,
            )

    observed_sha256 = file_sha256(STAGING_ARCHIVE_PATH)
    if observed_sha256 != expected_archive_sha256:
        raise RuntimeError(
            "recovered archive SHA-256 differs from the frozen source authority: "
            f"expected={expected_archive_sha256} observed={observed_sha256}"
        )


def validate_recovered_archive(
    inventory: list[dict[str, object]],
) -> None:
    expected_by_path = {
        str(entry["path"]): entry
        for entry in inventory
    }

    with zipfile.ZipFile(STAGING_ARCHIVE_PATH) as archive:
        members = tuple(archive.infolist())
        names = tuple(member.filename for member in members)

        if len(names) != len(set(names)):
            raise RuntimeError("recovered archive contains duplicate members")
        if names != tuple(sorted(expected_by_path)):
            raise RuntimeError("recovered archive member order or path set drifted")

        for member in members:
            relative_path = normalized_relative_path(member.filename).as_posix()
            expected = expected_by_path[relative_path]

            if member.flag_bits & 0x1:
                raise RuntimeError("recovered archive contains an encrypted member")
            if member.is_dir():
                raise RuntimeError("recovered archive contains a directory member")
            if member.filename.lower().endswith(ARCHIVE_SUFFIXES):
                raise RuntimeError("recovered archive contains a nested archive")

            unix_mode = member.external_attr >> 16
            expected_mode = 0o100755 if expected["executable"] else 0o100644
            if stat.S_IFMT(unix_mode) != stat.S_IFREG or unix_mode != expected_mode:
                raise RuntimeError("recovered archive member mode drifted")
            if member.date_time != ZIP_TIMESTAMP:
                raise RuntimeError("recovered archive member timestamp drifted")
            if member.compress_type != zipfile.ZIP_DEFLATED:
                raise RuntimeError("recovered archive compression method drifted")
            if member.file_size != expected["size_bytes"]:
                raise RuntimeError("recovered archive member size drifted")

            payload = archive.read(member)
            if bytes_sha256(payload) != expected["sha256"]:
                raise RuntimeError("recovered archive member identity drifted")


def extract_archive(
    inventory: list[dict[str, object]],
) -> None:
    expected_by_path = {
        str(entry["path"]): entry
        for entry in inventory
    }

    STAGING_HARNESS_ROOT.mkdir(parents=True)

    with zipfile.ZipFile(STAGING_ARCHIVE_PATH) as archive:
        for member in archive.infolist():
            relative_path = normalized_relative_path(member.filename)
            destination = STAGING_HARNESS_ROOT.joinpath(*relative_path.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)

            with archive.open(member, "r") as source, destination.open("xb") as target:
                shutil.copyfileobj(source, target, length=1024 * 1024)

            expected = expected_by_path[relative_path.as_posix()]
            destination.chmod(0o755 if expected["executable"] else 0o644)


def inspect_materialized_tree(
    inventory: list[dict[str, object]],
    receipt: dict[str, object],
) -> tuple[list[dict[str, object]], int]:
    observed_entries: list[dict[str, object]] = []
    observed_total_bytes = 0

    for path in sorted(
        STAGING_HARNESS_ROOT.rglob("*"),
        key=lambda item: item.as_posix(),
    ):
        if path.is_symlink():
            raise RuntimeError("materialized harness contains a symbolic link")

        metadata = path.stat()
        if path.is_dir():
            continue
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError("materialized harness contains a non-regular member")

        relative_path = path.relative_to(STAGING_HARNESS_ROOT).as_posix()
        if relative_path.lower().endswith(ARCHIVE_SUFFIXES):
            raise RuntimeError("materialized harness contains a nested archive")

        observed_total_bytes += metadata.st_size
        observed_entries.append(
            {
                "path": relative_path,
                "sha256": file_sha256(path),
                "size_bytes": metadata.st_size,
            }
        )

    expected_entries = [
        {
            "path": entry["path"],
            "sha256": entry["sha256"],
            "size_bytes": entry["size_bytes"],
        }
        for entry in inventory
    ]

    if observed_entries != expected_entries:
        raise RuntimeError("materialized harness inventory drifted")
    if len(observed_entries) != receipt["file_count"]:
        raise RuntimeError("materialized harness file count drifted")
    if observed_total_bytes != receipt["total_bytes"]:
        raise RuntimeError("materialized harness total bytes drifted")

    observed_directory_sha256 = bytes_sha256(
        canonical_json(observed_entries).encode("utf-8")
    )
    if observed_directory_sha256 != receipt["directory_sha256"]:
        raise RuntimeError("materialized harness directory identity drifted")

    return observed_entries, observed_total_bytes


def write_failure_evidence(exc: BaseException) -> None:
    if FAILURE_DIRECTORY.exists():
        shutil.rmtree(FAILURE_DIRECTORY)
    if FAILURE_ZIP_PATH.exists():
        FAILURE_ZIP_PATH.unlink()

    FAILURE_DIRECTORY.mkdir()
    failure = {
        "schema_version": "1.0.0",
        "materialization_status": "FAILED",
        "failure_class": "KAGGLE_EXPANDED_SOURCE_MATERIALIZATION_FAILED",
        "error_type": type(exc).__name__,
        "safe_message": str(exc)[:1000],
        "source_commit": EXPECTED_SOURCE_COMMIT,
        "input_dataset_name": DATASET_NAME,
        "network_access_performed": False,
        "package_installation_performed": False,
        "gpu_execution_performed": False,
        "model_loaded": False,
        "tokenizer_loaded": False,
        "worker_started": False,
        "model_requests_performed": 0,
        "authorization_issued": False,
    }
    failure_path = FAILURE_DIRECTORY / "90_failure.json"
    failure_path.write_text(canonical_json(failure), encoding="utf-8")

    evidence_manifest = {
        failure_path.name: file_sha256(failure_path),
    }
    manifest_path = FAILURE_DIRECTORY / "99_evidence_sha256.json"
    manifest_path.write_text(
        canonical_json(evidence_manifest),
        encoding="utf-8",
    )

    with zipfile.ZipFile(
        FAILURE_ZIP_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for path in (failure_path, manifest_path):
            info = zipfile.ZipInfo(path.name, date_time=ZIP_TIMESTAMP)
            info.create_system = 3
            info.compress_type = zipfile.ZIP_DEFLATED
            info.external_attr = 0o100644 << 16
            archive.writestr(
                info,
                path.read_bytes(),
                compress_type=zipfile.ZIP_DEFLATED,
                compresslevel=9,
            )


def materialize() -> dict[str, object]:
    if FINAL_BUNDLE_ROOT.exists() or STAGING_BUNDLE_ROOT.exists():
        raise RuntimeError(
            "materializer output or staging state already exists; "
            "run only in a fresh Kaggle session"
        )

    dataset_root = resolve_dataset_root()
    expanded_root = validate_dataset_topology(dataset_root)
    receipt, inventory, sha_manifest = load_and_validate_controls(dataset_root)
    by_path = validate_inventory(inventory, receipt)
    inspect_expanded_tree(expanded_root, by_path)

    STAGING_BUNDLE_ROOT.mkdir(parents=True)
    completed = False

    try:
        expected_archive_sha256 = sha_manifest[EXPECTED_ARCHIVE_NAME]
        reconstruct_exact_archive(
            expanded_root,
            inventory,
            expected_archive_sha256,
        )
        validate_recovered_archive(inventory)
        extract_archive(inventory)

        observed_entries, observed_total_bytes = inspect_materialized_tree(
            inventory,
            receipt,
        )
        observed_directory_sha256 = bytes_sha256(
            canonical_json(observed_entries).encode("utf-8")
        )

        materialization_receipt = {
            "schema_version": "1.0.0",
            "status": "CURRENT_CU129_HARNESS_MATERIALIZED",
            "producer_notebook_name": NOTEBOOK_NAME,
            "producer_output_directory": PRODUCER_OUTPUT_DIRECTORY,
            "source_commit": EXPECTED_SOURCE_COMMIT,
            "input_dataset_name": DATASET_NAME,
            "input_mode": "kaggle_expanded_source_recovered_to_exact_archive",
            "archive_filename": EXPECTED_ARCHIVE_NAME,
            "archive_sha256": expected_archive_sha256,
            "source_inventory_sha256": receipt["inventory_sha256"],
            "source_receipt_sha256": sha_manifest[
                "source_packaging_receipt.json"
            ],
            "source_sha256_manifest_sha256": file_sha256(
                dataset_root / "sha256_manifest.json"
            ),
            "output_directory": EXPECTED_OUTPUT_DIRECTORY,
            "directory_sha256": observed_directory_sha256,
            "file_count": len(observed_entries),
            "total_bytes": observed_total_bytes,
            "nested_archives_present": False,
            "symlinks_present": False,
            "network_access_performed": False,
            "package_installation_performed": False,
            "gpu_execution_performed": False,
            "model_loaded": False,
            "tokenizer_loaded": False,
            "worker_started": False,
            "model_requests_performed": 0,
            "benchmark_trajectory_requests_performed": 0,
            "authorization_issued": False,
            "kaggle_auto_expanded_source_detected": True,
            "exact_archive_reconstructed": True,
        }

        STAGING_RECEIPT_PATH.write_text(
            canonical_json(materialization_receipt),
            encoding="utf-8",
        )

        STAGING_ARCHIVE_PATH.unlink()
        STAGING_BUNDLE_ROOT.replace(FINAL_BUNDLE_ROOT)
        completed = True
        return materialization_receipt
    finally:
        if not completed and STAGING_BUNDLE_ROOT.exists():
            shutil.rmtree(STAGING_BUNDLE_ROOT)


if len(NOTEBOOK_NAME) > 50:
    raise RuntimeError("Kaggle notebook name exceeds 50 characters")

expected_failures = (
    RuntimeError,
    OSError,
    ValueError,
    TypeError,
    KeyError,
    zipfile.BadZipFile,
)

try:
    result = materialize()
except expected_failures as exc:
    write_failure_evidence(exc)
    print("materialization_status=FAILED")
    print("failure_class=KAGGLE_EXPANDED_SOURCE_MATERIALIZATION_FAILED")
    print(f"error_type={type(exc).__name__}")
    print(f"safe_message={str(exc)[:500]}")
    print("gpu_execution_performed=false")
    print("package_installation_performed=false")
    print("model_requests_performed=0")
    print("authorization_issued=false")
    print(f"failure_evidence_zip={FAILURE_ZIP_PATH}")
    raise

print("status=CURRENT_CU129_HARNESS_MATERIALIZED")
print(f"producer_output_directory={PRODUCER_OUTPUT_DIRECTORY}")
print(f"source_commit={EXPECTED_SOURCE_COMMIT}")
print(f"output_directory={EXPECTED_OUTPUT_DIRECTORY}")
print(f"file_count={result['file_count']}")
print(f"total_bytes={result['total_bytes']}")
print(f"directory_sha256={result['directory_sha256']}")
print("input_mode=kaggle_expanded_source_recovered_to_exact_archive")
print("exact_archive_reconstructed=true")
print("gpu_execution_performed=false")
print("package_installation_performed=false")
print("model_requests_performed=0")
print("authorization_issued=false")
print("save_this_notebook_output=true")
